In [ ]:
# Step 1 — Membership inference with a lightweight UPMASK-style voting scheme
#
# Purpose
#   - Load a VizieR-style TSV/CSV (semicolon-delimited) table containing Gaia astrometry.
#   - Standardize column names, coerce numeric types, and drop rows with missing required fields.
#   - Run a Monte Carlo + KMeans voting procedure to assign membership probabilities.
#   - Save the final member catalog and generate sanity-check plots.
#
# Notes
#   - This is a simplified, reproducible approximation inspired by UPMASK-like resampling + clustering.
#   - The "best cluster" in each iteration is selected by proximity to a literature target (pmra, pmdec, parallax).
#   - Parameters below are intended to be edited per cluster.

import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler


warnings.filterwarnings("ignore", category=UserWarning)


# ---------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------
@dataclass
class ClusterConfig:
    cluster_name: str
    input_path: str

    # Output
    output_members_csv: str = "Members_Final.csv"

    # Parsing (VizieR exports are commonly ';' delimited with '#' comments)
    sep: str = ";"
    comment: str = "#"

    # UPMASK-lite settings
    iterations: int = 50
    k_clusters: int = 10
    membership_threshold: float = 0.50

    # Target literature values (edit per cluster)
    target_pmra: float = -10.9
    target_pmdec: float = -2.9
    target_parallax: float = 1.13

    # Weighting for selecting the "best cluster" in each iteration
    # Increase parallax_weight if parallax is more reliable than PMs in your sample.
    parallax_weight: float = 10.0

    # Feature columns used for clustering (must exist after renaming)
    feature_cols: Tuple[str, ...] = ("ra", "dec", "parallax", "pmra", "pmdec")

    # Error columns used for Monte Carlo perturbation
    error_cols: Tuple[str, ...] = ("parallax_error", "pmra_error", "pmdec_error")

    # Plot controls (edit per cluster if needed)
    vpd_xlim: Tuple[float, float] = (-25.0, 5.0)
    vpd_ylim: Tuple[float, float] = (-15.0, 5.0)

    # Optional: reference lines in VPD (set to None to disable)
    vpd_ref_pmra: Optional[float] = -10.9
    vpd_ref_pmdec: Optional[float] = -2.9


# ---------------------------------------------------------------------
# Column standardization
# ---------------------------------------------------------------------
DEFAULT_RENAME_MAP: Dict[str, str] = {
    # Common VizieR/Gaia-style names -> standardized names
    "RA_ICRS": "ra",
    "DE_ICRS": "dec",
    "Plx": "parallax",
    "e_Plx": "parallax_error",
    "pmRA": "pmra",
    "e_pmRA": "pmra_error",
    "pmDE": "pmdec",
    "e_pmDE": "pmdec_error",
    "Gmag": "phot_g_mean_mag",
    "BP-RP": "bp_rp",
    "RUWE": "ruwe",
}


# ---------------------------------------------------------------------
# Utilities
# ---------------------------------------------------------------------
def load_and_clean_table(
    path: str,
    sep: str = ";",
    comment: str = "#",
    rename_map: Optional[Dict[str, str]] = None,
    required_numeric: Iterable[str] = ("ra", "dec", "pmra", "pmdec", "parallax", "parallax_error"),
) -> pd.DataFrame:
    """
    Load a semicolon-delimited table (VizieR-like), standardize columns, coerce numeric types,
    and drop rows missing required fields.
    """
    path_obj = Path(path)
    if not path_obj.exists():
        raise FileNotFoundError(f"Input file not found: {path_obj.resolve()}")

    df = pd.read_csv(path_obj, sep=sep, comment=comment, on_bad_lines="skip")
    if rename_map:
        df = df.rename(columns=rename_map)

    # Coerce relevant columns to numeric if present
    for col in set(required_numeric).union({"ruwe", "phot_g_mean_mag", "bp_rp", "pmra_error", "pmdec_error"}):
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # Drop rows lacking required numeric fields
    missing = [c for c in required_numeric if c not in df.columns]
    if missing:
        raise KeyError(f"Missing required columns after renaming: {missing}")

    df_clean = df.dropna(subset=list(required_numeric)).copy()
    return df_clean


def plot_initial_sanity_checks(df: pd.DataFrame, cfg: ClusterConfig) -> None:
    """Plot sky distribution and vector-point diagram before membership selection."""
    fig = plt.figure(figsize=(12, 5))

    # Sky map
    ax1 = fig.add_subplot(1, 2, 1)
    ax1.scatter(df["ra"], df["dec"], s=1, alpha=0.1)
    ax1.set_title(f"{cfg.cluster_name}: Initial Sky Distribution (N={len(df)})")
    ax1.set_xlabel("RA (deg)")
    ax1.set_ylabel("Dec (deg)")
    ax1.invert_xaxis()
    ax1.grid(True, alpha=0.3)

    # VPD
    ax2 = fig.add_subplot(1, 2, 2)
    ax2.scatter(df["pmra"], df["pmdec"], s=1, alpha=0.1)
    ax2.set_title("Vector Point Diagram (Pre-selection)")
    ax2.set_xlabel("pmRA (mas yr$^{-1}$)")
    ax2.set_ylabel("pmDec (mas yr$^{-1}$)")
    ax2.set_xlim(*cfg.vpd_xlim)
    ax2.set_ylim(*cfg.vpd_ylim)
    ax2.grid(True, alpha=0.3)

    if cfg.vpd_ref_pmra is not None:
        ax2.axvline(cfg.vpd_ref_pmra, linestyle="--", linewidth=0.8)
    if cfg.vpd_ref_pmdec is not None:
        ax2.axhline(cfg.vpd_ref_pmdec, linestyle="--", linewidth=0.8)

    plt.tight_layout()
    plt.show()


def upmask_lite_membership(df: pd.DataFrame, cfg: ClusterConfig) -> pd.DataFrame:
    """
    Compute membership probabilities via repeated resampling + KMeans clustering.

    Returns a copy of df with an added column: 'prob' (membership probability in [0, 1]).
    """
    df = df.copy()
    n = len(df)

    # Validate necessary columns
    for col in cfg.feature_cols:
        if col not in df.columns:
            raise KeyError(f"Missing feature column '{col}' in input data.")
    for col in cfg.error_cols:
        if col not in df.columns:
            raise KeyError(f"Missing error column '{col}' in input data.")

    X_cols = list(cfg.feature_cols)
    votes = np.zeros(n, dtype=float)

    for i in range(cfg.iterations):
        # Resampling (Gaussian noise using quoted uncertainties)
        X_noisy = df[X_cols].copy()
        X_noisy["parallax"] += df["parallax_error"].values * np.random.randn(n)
        X_noisy["pmra"] += df["pmra_error"].values * np.random.randn(n)
        X_noisy["pmdec"] += df["pmdec_error"].values * np.random.randn(n)

        # Scaling
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X_noisy)

        # KMeans (fixed seed per iteration for reproducibility)
        kmeans = KMeans(n_clusters=cfg.k_clusters, n_init=10, random_state=i)
        labels = kmeans.fit_predict(X_scaled)

        # Convert cluster centers back to original feature space
        centers_scaled = kmeans.cluster_centers_
        centers = scaler.inverse_transform(centers_scaled)

        # centers column order corresponds to cfg.feature_cols
        # feature_cols = ("ra","dec","parallax","pmra","pmdec")
        # Build a distance metric to the target literature astrometry
        pmra_idx = X_cols.index("pmra")
        pmdec_idx = X_cols.index("pmdec")
        plx_idx = X_cols.index("parallax")

        dist_sq = (
            (centers[:, pmra_idx] - cfg.target_pmra) ** 2
            + (centers[:, pmdec_idx] - cfg.target_pmdec) ** 2
            + cfg.parallax_weight * (centers[:, plx_idx] - cfg.target_parallax) ** 2
        )
        best_cluster_id = int(np.argmin(dist_sq))

        # Vote
        votes[labels == best_cluster_id] += 1.0

        if (i + 1) % max(1, cfg.iterations // 5) == 0:
            print(f"Iteration {i + 1:>3d}/{cfg.iterations} completed.")

    df["prob"] = votes / float(cfg.iterations)
    return df


def plot_membership_results(df_with_prob: pd.DataFrame, cfg: ClusterConfig) -> None:
    """Plot members vs field in sky and proper motion space, plus probability histogram."""
    members = df_with_prob[df_with_prob["prob"] >= cfg.membership_threshold]
    field = df_with_prob[df_with_prob["prob"] < cfg.membership_threshold]

    print("\nMembership summary")
    print(f"  Total sources processed: {len(df_with_prob)}")
    print(f"  Likely members (P >= {cfg.membership_threshold:.2f}): {len(members)}")
    print(f"  Field contaminants (P < {cfg.membership_threshold:.2f}): {len(field)}")

    fig = plt.figure(figsize=(12, 6))

    # Sky map
    ax1 = fig.add_subplot(1, 2, 1)
    ax1.scatter(field["ra"], field["dec"], s=1, alpha=0.1, label="Field")
    ax1.scatter(members["ra"], members["dec"], s=5, alpha=0.8, label="Members")
    ax1.set_title(f"{cfg.cluster_name}: Membership Spatial Distribution")
    ax1.set_xlabel("RA (deg)")
    ax1.set_ylabel("Dec (deg)")
    ax1.invert_xaxis()
    ax1.grid(True, alpha=0.3)
    ax1.legend(loc="upper right", frameon=True)

    # VPD
    ax2 = fig.add_subplot(1, 2, 2)
    ax2.scatter(field["pmra"], field["pmdec"], s=1, alpha=0.1)
    ax2.scatter(members["pmra"], members["pmdec"], s=5, alpha=0.6)
    ax2.set_title("Membership in Proper Motion Space")
    ax2.set_xlabel("pmRA (mas yr$^{-1}$)")
    ax2.set_ylabel("pmDec (mas yr$^{-1}$)")
    ax2.set_xlim(*cfg.vpd_xlim)
    ax2.set_ylim(*cfg.vpd_ylim)
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    # Probability histogram
    plt.figure(figsize=(6, 4))
    plt.hist(df_with_prob["prob"], bins=20, edgecolor="black")
    plt.title("Membership Probability Distribution")
    plt.xlabel("Membership probability")
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()


def main(cfg: ClusterConfig) -> None:
    print(f"Loading input table: {cfg.input_path}")
    df_clean = load_and_clean_table(
        path=cfg.input_path,
        sep=cfg.sep,
        comment=cfg.comment,
        rename_map=DEFAULT_RENAME_MAP,
        required_numeric=("ra", "dec", "pmra", "pmdec", "parallax", "parallax_error", "pmra_error", "pmdec_error"),
    )
    print(f"Data cleaned: {len(df_clean)} rows retained.")

    plot_initial_sanity_checks(df_clean, cfg)

    print(f"\nRunning membership inference (iterations={cfg.iterations}, k_clusters={cfg.k_clusters})")
    df_prob = upmask_lite_membership(df_clean, cfg)

    plot_membership_results(df_prob, cfg)

    members = df_prob[df_prob["prob"] >= cfg.membership_threshold].copy()
    out_path = Path(cfg.output_members_csv)
    members.to_csv(out_path, index=False)
    print(f"\nSaved member catalog: {out_path.resolve()}")


# ---------------------------------------------------------------------
# User configuration (edit here)
# ---------------------------------------------------------------------
# Example: M67
cfg = ClusterConfig(
    cluster_name="M67",
    input_path="m67.tsv",
    output_members_csv="M67_Members_Final.csv",
    iterations=50,
    k_clusters=10,
    membership_threshold=0.50,
    target_pmra=-10.9,
    target_pmdec=-2.9,
    target_parallax=1.13,
    parallax_weight=10.0,
    vpd_xlim=(-25, 5),
    vpd_ylim=(-15, 5),
    vpd_ref_pmra=-10.9,
    vpd_ref_pmdec=-2.9,
)

main(cfg)
